# Strategy health check clock correction practice — 2026-10-03

本 notebook 只展示已保存的异步研究结果，不重新计算策略。该研究模型不是原生产契约修复，也不代表券商成交、换汇或结算验证。

参见[修正报告](../../docs/notes/strategy-healthcheck-clock-correction-2026-10-03-review.md)和[输出 README](../../outputs/strategy-healthcheck-clock-correction-2026-10-03/README.md)。BTC UTC 日线可用时点、分市场成交、现金分组限制和滞后 FX 记账约定均见 README。


In [ ]:
import os
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "xquant").exists():
            return candidate
    raise RuntimeError("repository root not found")

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
OUTPUT_DIR = REPO_ROOT / "outputs" / "strategy-healthcheck-clock-correction-2026-10-03"


In [ ]:
from pathlib import Path
import gzip
import json
import pandas as pd
try:
    from IPython.display import display, Markdown
except ImportError:
    class Markdown(str):
        pass

    def display(value):
        if isinstance(value, pd.DataFrame):
            print(f"DataFrame: {value.shape[0]} rows × {value.shape[1]} columns")
            print(value.head(5).to_string(index=False))
        elif isinstance(value, pd.Series):
            print(value.to_string())
        else:
            print(value)

def show_csv(relative_path, title=None):
    path = OUTPUT_DIR / relative_path
    display(Markdown(f"### {title or relative_path}"))
    if not path.exists():
        display(Markdown(f"_尚未生成：`{path.relative_to(REPO_ROOT)}`_"))
        return None
    frame = pd.read_csv(path)
    display(frame)
    return frame

def show_json(relative_path, title=None):
    path = OUTPUT_DIR / relative_path
    display(Markdown(f"### {title or relative_path}"))
    if not path.exists():
        display(Markdown(f"_尚未生成：`{path.relative_to(REPO_ROOT)}`_"))
        return None
    value = json.loads(path.read_text())
    display(value)
    return value


## BTC 异步执行结果

以下表格读取 runner/finalize 保存的结果。窗口指标、额外成本、人民币估值敏感性仍对应一个理想化分数份额和即时结算研究模型。


In [ ]:
btc_summary = show_csv("btc/summary.csv", "BTC summary")
btc_cost = show_csv("btc/cost_sensitivity.csv", "BTC cost sensitivity")
show_csv("btc/annual_returns.csv", "BTC annual returns")
show_csv("btc/monthly_returns.csv", "BTC monthly returns")
show_csv("btc/monthly_stats.csv", "BTC monthly statistics")
show_csv("btc/rolling_3y_5y.csv", "BTC rolling 3y/5y returns")
fx = show_json("btc/fx_diagnostic.json", "Lagged FX accounting sensitivity")
if fx is not None:
    print("FX assumption:", fx.get("assumption"))
    display(pd.DataFrame([{"window": name, **metrics} for name, metrics in fx.items() if isinstance(metrics, dict)]))


## 严格 Q3 相关性

此表应由 finalize 按 `2026-06-30 < date <= 2026-09-30` 重新生成；64 个样本是不同市场的日标签相关性，不代表同步收盘时点相关性。


In [ ]:
show_csv("btc_correlations.csv", "BTC / US asset Q3 return correlations")


## Multi-asset 异步结果

完整 JSON 可能很大或仍在写入。本单元只读取并显示推荐状态、覆盖范围、稳定性门槛和目标候选摘要；不把完整 JSON 展开。读取过程中若压缩文件尚未完成，会提示等待 finalize 输入就绪后重试。


In [ ]:
multi_path = OUTPUT_DIR / "multi_asset_full.json.gz"
if not multi_path.exists():
    display(Markdown("_Multi-asset 结果尚未出现；请等待 family runner 完成。_"))
else:
    try:
        with gzip.open(multi_path, "rt", encoding="utf-8") as stream:
            multi = json.load(stream)
    except (EOFError, OSError, json.JSONDecodeError) as exc:
        display(Markdown(f"_Multi-asset 文件存在但尚未完整可读（{type(exc).__name__}）；等待 runner 完成后重试。_"))
    else:
        recommendation = multi.get("recommendation", {})
        display(Markdown("### Recommendation"))
        display(pd.DataFrame([{
            "status": recommendation.get("status"),
            "candidateId": recommendation.get("candidateId"),
            "researchLeadId": recommendation.get("researchLeadId"),
            "reason": recommendation.get("reason"),
        }]))

        coverage = multi.get("coverage", {})
        display(Markdown("### Coverage"))
        if isinstance(coverage, dict):
            by_symbol = coverage.get("bySymbol", {})
            display(pd.DataFrame([{
                "commonStartDate": coverage.get("commonStartDate"),
                "commonEndDate": coverage.get("commonEndDate"),
                "symbols": len(by_symbol) if isinstance(by_symbol, dict) else None,
                "missingSymbols": coverage.get("missingSymbols"),
                "missingStressPeriods": coverage.get("missingStressPeriods"),
                "fxSymbol": coverage.get("fxSymbol"),
                "fxStartDate": coverage.get("fxStartDate"),
                "fxEndDate": coverage.get("fxEndDate"),
            }]))
        else:
            display(coverage)

        display(Markdown("### Stability"))
        stability = multi.get("stability", {})
        display(pd.DataFrame([stability]) if isinstance(stability, dict) else pd.DataFrame(stability))

        candidate_id = recommendation.get("candidateId")
        candidate = next((row for row in multi.get("variants", []) if row.get("id") == candidate_id), None)
        counts = {
            "variants": len(multi.get("variants", [])),
            "benchmarks": len(multi.get("benchmarks", [])),
            "cost sensitivity rows": len(multi.get("costSensitivity", [])),
            "FX sensitivity rows": len(multi.get("fxSensitivity", [])),
            "parameter matrix rows": len(multi.get("parameterMatrix", [])),
            "static weight stability rows": len(multi.get("staticWeightStability", [])),
        }
        display(Markdown("### Result summary (counts and one candidate only)"))
        display(pd.Series(counts, name="rows"))
        if candidate is not None:
            display(pd.DataFrame([{"id": candidate.get("id"), **candidate.get("summary", {})}]))
        else:
            display(Markdown("_Recommendation candidate is not present in variants._"))


## Reproducibility and scope

Use the unmodified 83-file input snapshot under `outputs/strategy-healthcheck-2026-10-02/market_data/`. Re-running into the canonical correction output is unsafe for BTC while its `btc/` directory exists. Share the original `market_data/`, the finalized `source_bundle.zip`, and configurations with another reviewer; see the [README](../../outputs/strategy-healthcheck-clock-correction-2026-10-03/README.md).
